# Cooling-system optimization

Regenerate the original thermal model and run its one-phase-bit mock optimization.

Set up the local quantum and thermal experiment.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq
from examples.synthesis.applications.automotive.cooling_systems_optimization.overview import full_model, model, quantum, workflow
from examples.synthesis.applications.automotive.cooling_systems_optimization.overview.main import print_report, save_figures

Use the native amplitude preparation, block encoding and phase fitting.

In [ ]:
print(fq.__name__)

Retain the original ambient temperature, heats and six resistances.

In [ ]:
print('Ambient and heat inputs:', model.AMBIENT, model.HEAT)
print('Thermal resistances:', model.RESISTANCES)

Regenerate the four-node matrix from its conductance toggles.

In [ ]:
all_links = model.build_matrix(63)
print(all_links)

Inspect the all-links thermal system.

In [ ]:
assert tuple(all_links.shape) == (4, 4)

Build the original pair subblocks and selector-based LCU.

In [ ]:
pairs = [quantum.pair_block(i, j, enabled=True) for i, j in model.EDGES]
assert len(pairs) == 6

Retain the original three selector bits, two block bits and two data bits.

In [ ]:
print('Thermal block and data widths:', quantum.THERMAL_BLOCK, quantum.THERMAL_DATA)

Execute and verify the complete thermal and mock workflows.

In [ ]:
result = workflow.run()

Report the original LCU normalization and fresh physical matrix.

In [ ]:
print('Conductance normalization:', model.alpha())
print(result.matrix)

Verify the independently constructed encoded block.

In [ ]:
assert result.matrix_error < 1e-10

Fit the standalone degree-11 reciprocal demonstration on [1/3, 1].

In [ ]:
assert result.demo_reciprocal.fit.phases.numel() == 12
figures = Path('notebook-figures')
save_figures(result, figures)
display(Image(filename=str(figures / 'thermal-inverse.png')))

Keep the full thermal inverse, separate from the standalone reciprocal demonstration.

In [ ]:
assert result.reciprocal.minimum == 1 / (model.alpha() * model.R_ENV)

Prepare the original signed four-entry heat vector.

In [ ]:
assert len(model.HEAT) == 4 and min(model.HEAT) < 0

Run all eleven inverse queries and retain the measured circuit width.

In [ ]:
print('Inverse resource estimate:', result.inverse_resources)

Check the projected amplitudes against the independent diagonalization.

In [ ]:
assert result.inverse_error < 1e-10

Read the signed updates and the original temperature magnitudes.

In [ ]:
print('QSVT magnitudes:', result.computed.abs().tolist())
print('Classical magnitudes:', result.reference.abs().tolist())
assert result.relative_error < 0.04

Preserve the prescribed phase function sin(pi*y/32) on the QAE register.

In [ ]:
print('One-bit phase slope:', quantum.QAE_PHASE_WEIGHT)

Validate all 64 configurations of the coherent thermal oracle.

In [ ]:
print('Conditional error:', result.conditional_error)
assert result.conditional_error < 1e-10

Define the original-sized five-bit QAE and full design model.

In [ ]:
def full_thermal_design(gamma: float, beta: float) -> fq.Circuit:
    return full_model.build_full_quso(result.demo_reciprocal, gamma, beta)

Keep the full 20-bit QuSO run disabled, as in the source.

In [ ]:
print('Full design width when requested:', full_model.WIDTH)
assert full_model.WIDTH == 20

Load the original 64 mock costs only for the separate one-bit circuit.

In [ ]:
assert result.costs.numel() == 64
print('Mock QAE costs:', result.costs.tolist())

Use one symbolic mock model for two finite nonzero angle pairs.

In [ ]:
print('Mock angle bindings:', workflow.BINDINGS)
assert result.mock_probabilities.shape == (2, 64)

Report actual block, inverse and mock-circuit validation results.

In [ ]:
print_report(result)